# batchnorm-affine-params composite — cx28: BatchNorm2d forward with both affine params and running stats

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `batchnorm-affine-params`, `batchnorm-running-stats`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "batchnorm-affine-params"
DD_ATOM_IDS = ["batchnorm-affine-params", "batchnorm-running-stats"]
DD_SUBTOPICS = ["CNN: BatchNorm affine params", "CNN: BatchNorm running stats"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

A BatchNorm layer carries TWO kinds of state:
1. **Affine params** — `gamma` (a.k.a. `weight`) and `beta` (`bias`), both shape `(C,)`. These are LEARNED via SGD. They scale and shift the normalised activations: `y = gamma * (x - mu) / sqrt(var + eps) + beta`. They are wrapped in `nn.Parameter` so the optimizer sees them.
2. **Running stats** — `running_mean` and `running_var`, also shape `(C,)`. These are NOT parameters — they are *buffers*, updated by a moving average over training batches and used unchanged at inference. They must be registered with `self.register_buffer(...)` so `.to(device)` moves them and `.state_dict()` saves them, but the optimizer does NOT see them.

**Anatomy of a manual BN forward (train mode).**
```python
mu = x.mean(dim=(0, 2, 3))                     # (C,) per-channel batch mean.
var = x.var(dim=(0, 2, 3), unbiased=False)     # (C,) per-channel batch var.
# Update running stats in-place with momentum.
self.running_mean.mul_(1 - momentum).add_(mu * momentum)
self.running_var.mul_(1 - momentum).add_(var * momentum)
# Normalize using BATCH stats during training.
x_hat = (x - mu[None, :, None, None]) / t.sqrt(var[None, :, None, None] + eps)
# Apply learned affine — gamma * x_hat + beta.
return self.weight[None, :, None, None] * x_hat + self.bias[None, :, None, None]
```

**Why both atoms together.** Affine without running stats = a normalised activation with no inference-time stand-in for the batch mean/var. Running stats without affine = no way for the model to learn to undo the normalisation when that's what helps. They MUST coexist; one is `Parameter`, the other is `buffer`.

### Composite Exercise — BatchNorm2d forward with both affine params and running stats

**Atoms exercised together**: `batchnorm-affine-params`, `batchnorm-running-stats`

Implement `cx28_make_batchnorm2d()` — return the class `MyBatchNorm2d(nn.Module)`.

Required structure:
- `__init__(self, num_features, eps=1e-5, momentum=0.1)`:
  - `super().__init__()`
  - `self.eps = eps; self.momentum = momentum`
  - Affine params (atom: batchnorm-affine-params):
    - `self.weight = nn.Parameter(t.ones(num_features))`
    - `self.bias   = nn.Parameter(t.zeros(num_features))`
  - Running stats (atom: batchnorm-running-stats — REGISTER as BUFFERS):
    - `self.register_buffer('running_mean', t.zeros(num_features))`
    - `self.register_buffer('running_var',  t.ones(num_features))`
- `forward(self, x)` — assume `self.training is True` (we test eval-vs-train in cx29):
  - Compute per-channel `mu`, `var` over `(N, H, W)` (i.e. `dim=(0, 2, 3)`, `unbiased=False`).
  - Update `self.running_mean` and `self.running_var` in-place with momentum (formula: `new = (1 - m) * old + m * batch`).
  - Normalize x using the BATCH stats, apply affine. Return the result.

The test checks: (a) `weight` and `bias` are Parameters; (b) `running_mean` and `running_var` are in `state_dict` but NOT in `parameters()`; (c) the forward output equals `F.batch_norm(x, running_mean, running_var, weight, bias, training=True, momentum=...)`; (d) after the forward, `running_mean` and `running_var` were updated by the momentum rule.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx28_make_batchnorm2d():
    """Return the MyBatchNorm2d class."""
    raise NotImplementedError

def _test_cx28():
    MyBN = cx28_make_batchnorm2d()
    assert issubclass(MyBN, nn.Module)

    # Case A: affine params are Parameters; running stats are buffers (NOT parameters).
    t.manual_seed(0)
    bn = MyBN(num_features=4)
    param_names = {name for name, _ in bn.named_parameters()}
    buffer_names = {name for name, _ in bn.named_buffers()}
    assert 'weight' in param_names and 'bias' in param_names, (
        f'weight & bias must be nn.Parameter; param names = {param_names}'
    )
    assert 'running_mean' in buffer_names and 'running_var' in buffer_names, (
        f'running_mean & running_var must be registered buffers; buffer names = {buffer_names}'
    )
    # Running stats must NOT appear in parameters — that would let SGD overwrite them.
    assert 'running_mean' not in param_names, 'running_mean is a buffer, not a Parameter'
    assert 'running_var' not in param_names, 'running_var is a buffer, not a Parameter'

    # Case B: initial values — weight=1, bias=0, running_mean=0, running_var=1.
    assert t.allclose(bn.weight, t.ones(4))
    assert t.allclose(bn.bias, t.zeros(4))
    assert t.allclose(bn.running_mean, t.zeros(4))
    assert t.allclose(bn.running_var, t.ones(4))

    # Case C: forward output equals F.batch_norm reference.
    t.manual_seed(1)
    bn = MyBN(num_features=3, eps=1e-5, momentum=0.1)
    # Make affine params non-trivial so the affine half actually matters.
    with t.no_grad():
        bn.weight.copy_(t.tensor([0.5, 2.0, 1.5]))
        bn.bias.copy_(t.tensor([0.1, -0.2, 0.3]))
        bn.running_mean.copy_(t.tensor([0.0, 0.0, 0.0]))
        bn.running_var.copy_(t.tensor([1.0, 1.0, 1.0]))
    # Save copies of running stats BEFORE the forward (which mutates them).
    rm_before = bn.running_mean.clone()
    rv_before = bn.running_var.clone()
    x = t.randn(2, 3, 4, 5)
    # Reference computed with F.batch_norm (training=True so it updates the stats we pass).
    rm_ref = rm_before.clone()
    rv_ref = rv_before.clone()
    ref = F.batch_norm(x, rm_ref, rv_ref, bn.weight, bn.bias, training=True, momentum=0.1, eps=1e-5)
    out = bn(x)
    assert tuple(out.shape) == (2, 3, 4, 5)
    assert t.allclose(out, ref, atol=1e-5), (
        f'forward output mismatch with F.batch_norm reference; max err = {(out - ref).abs().max().item()}'
    )

    # Case D: running stats were updated by the momentum rule.
    assert not t.allclose(bn.running_mean, rm_before), 'running_mean must update during train forward'
    assert not t.allclose(bn.running_var, rv_before), 'running_var must update during train forward'
    # Cross-check the exact updated values match the reference.
    assert t.allclose(bn.running_mean, rm_ref, atol=1e-5), 'running_mean update rule mismatch'
    assert t.allclose(bn.running_var, rv_ref, atol=1e-5), 'running_var update rule mismatch'
    _dd_passed.add('cx28')

_test_cx28()

<details><summary>Show solution — cx28</summary>

```python
def cx28_make_batchnorm2d():
    class MyBatchNorm2d(nn.Module):
        def __init__(self, num_features, eps=1e-5, momentum=0.1):
            super().__init__()
            self.eps = eps
            self.momentum = momentum
            # Atom A (batchnorm-affine-params): learned scale gamma and shift beta.
            self.weight = nn.Parameter(t.ones(num_features))
            self.bias = nn.Parameter(t.zeros(num_features))
            # Atom B (batchnorm-running-stats): tracked via register_buffer — moves with .to(),
            # saved by state_dict(), but the optimizer doesn't update them.
            self.register_buffer('running_mean', t.zeros(num_features))
            self.register_buffer('running_var', t.ones(num_features))

        def forward(self, x):
            # Per-channel batch statistics (reduce N, H, W; keep C).
            mu = x.mean(dim=(0, 2, 3))
            var = x.var(dim=(0, 2, 3), unbiased=False)
            # Momentum-update the buffers in-place.
            self.running_mean.mul_(1 - self.momentum).add_(mu.detach() * self.momentum)
            # Note: PyTorch uses UNBIASED var for the running stat update.
            n = x.shape[0] * x.shape[2] * x.shape[3]
            var_unbiased = var * (n / (n - 1)) if n > 1 else var
            self.running_var.mul_(1 - self.momentum).add_(var_unbiased.detach() * self.momentum)
            # Normalize with BATCH stats during training, then affine.
            x_hat = (x - mu[None, :, None, None]) / t.sqrt(var[None, :, None, None] + self.eps)
            return self.weight[None, :, None, None] * x_hat + self.bias[None, :, None, None]

    return MyBatchNorm2d
```

Two subtleties: (1) the batch var used for NORMALIZATION is biased (`unbiased=False`), but the var used to update `running_var` is UNBIASED — PyTorch's actual behaviour. (2) `register_buffer` is the canonical mechanism for non-trainable state on a Module. It's not just a `self.x = tensor` — registering puts the tensor in `state_dict()` and `to(device)`'s sweep, both of which `nn.Parameter` would also do, but with the crucial difference that buffers are skipped by the optimizer.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx28'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx28',
        'subtopics': ["CNN: BatchNorm affine params", "CNN: BatchNorm running stats"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()